In [0]:
%pip install pdfplumber

In [0]:
import io, pdfplumber

def profile_pdf(content):
    try:
        with pdfplumber.open(io.BytesIO(content)) as pdf:
            n_pages = len(pdf.pages)
            chars_per = [len(p.chars) for p in pdf.pages]
            chars = [c for p in pdf.pages for c in p.chars]
            rotations = sorted({int(p.rotation) for p in pdf.pages})
        n_chars = len(chars)
        return {
            "page_count": n_pages,
            "empty_pages": sum(1 for n in chars_per if n < 20),
            "chars_per_page": float(round(n_chars / n_pages, 1)) if n_pages else 0.0,
            "digit_ratio": float(round(sum(c["text"].isdigit() for c in chars) / n_chars, 3)) if n_chars else 0.0,
            "rotated_ratio": float(round(sum(not c.get("upright", True) for c in chars) / n_chars, 3)) if n_chars else 0.0,
            "cid_ratio": float(round(sum(c["text"].startswith("(cid:") for c in chars) / n_chars, 3)) if n_chars else 0.0,
            "page_rotations": rotations,
            "error": None,
        }
    except Exception as e:
        return {"page_count": None, "empty_pages": None, "chars_per_page": None,
                "digit_ratio": None, "rotated_ratio": None, "cid_ratio": None,
                "page_rotations": None, "error": str(e)[:500]}

In [0]:
from pyspark.sql.functions import udf, col, when, current_timestamp
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, ArrayType, StringType

profile_schema = StructType([
    StructField("page_count", IntegerType()),
    StructField("empty_pages", IntegerType()),
    StructField("chars_per_page", DoubleType()),
    StructField("digit_ratio", DoubleType()),
    StructField("rotated_ratio", DoubleType()),
    StructField("cid_ratio", DoubleType()),
    StructField("page_rotations", ArrayType(IntegerType())),
    StructField("error", StringType()),
])

profile_udf = udf(profile_pdf, profile_schema).asNondeterministic()   # run once per file

profile = (
    spark.table("tnelections.form20_bronze.pdf_files")
    .repartition(32)                                                    # spread files across workers
    .withColumn("p", profile_udf(col("content")))
    .select("year", "ac_no", "path", "p.*")
    .withColumn("read_method",
        when(col("error").isNotNull(), "review")
        .when(col("empty_pages") > 0, "ocr")
        .when(col("cid_ratio") > 0.05, "ocr")
        .otherwise("text"))
    .withColumn("read_reason",
        when(col("error").isNotNull(), "pdf could not be opened")
        .when(col("empty_pages") == col("page_count"), "no text layer")
        .when(col("empty_pages") > 0, "some pages have no text")
        .when(col("cid_ratio") > 0.05, "unmapped glyphs in text layer")
        .otherwise("text layer present"))
    .withColumn("profiled_at", current_timestamp())
)

(profile.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("tnelections.form20_bronze.pdf_profile"))

In [0]:
pdf_files = spark.table("tnelections.form20_bronze.pdf_files")

for ac in [16, 154]:
    row = pdf_files.filter(f"year = 2026 AND ac_no = {ac}").select("content").first()
    print(ac, profile_pdf(row.content))

In [0]:
display(spark.sql("""
    SELECT read_method, read_reason, COUNT(*) AS files, MAX(cid_ratio) AS max_cid_ratio
    FROM tnelections.form20_bronze.pdf_profile
    GROUP BY ALL
    ORDER BY files DESC
"""))